# Planted Noisy kXOR: Kikuchi Adjacency List

Native counterpart of the complete Qualtran v0.7.0 notebook at revision
`bcb17bb25360ed68ce3bc084645db3e929799389`, frozen blob
`7ed33e384065876e374a0eb63c88676f09c01b0e`. Copyright 2024 Google LLC, adapted under Apache-2.0.

The following mathematical description records the source intent. These roots
currently expose declarations and source resource estimates. Physical execution,
complex amplitudes and work-zero acceptance remain open in the corresponding
source audit. The adjacency-list root also preserves the documented source
return/free mismatch. Executable child components use `circuit.append(op)`.


In [ ]:
from collections.abc import Callable
from typing import cast

from IPython import display as notebook_display

from flagquantum.algorithms.kikuchi_adjacency import KikuchiNonZeroIndex
from flagquantum.algorithms.kxor_instance import example_kxor_instance
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(
        type(register.bitsize) is int
        and all(type(size) is int for size in register.shape_symbolic)
        for register in signature
    ):
        dot = export_signature_graph_dot(signature, name)
    else:
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " shape"] = str(register.shape_symbolic)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `KikuchiNonZeroIndex`
Adjacency list oracle $O_F$ for the Kikuchi matrix.

The oracle $O_F$ (Definition 4.5) takes in $i, k$,
and outputs $i, f(i, k)$ where $f(i, k)$ is
index of the $k$-th non-zero entry in row $i$.

As the Kikuchi matrix is symmetric, we can use the same oracle for both rows and columns.

The Kikuchi matrix is indexed by $S \in {[n] \choose k}$.
For a given row $S$ and column $T$, the entry $\mathcal{K}_{k}_{S, T}$
is potentially non-zero if $S \Delta T = U_j$ for some $j$, which is
equivalent to $T = S \Delta U_j$.
Here, $U_j$ is the $j$-th unique scope in the instance $\mathcal{I}$,
and $\Delta$ is the symmetric difference operator.

See docstring for :class:`KXorInstance` for the overall problem definition.

To find the $k$-th non-zero entry, we use two oracles:
1. $(S, k) \mapsto f(S, k)$, implemented by `ColumnOfKthNonZeroEntry`
2. $(S, f(S, k)) \mapsto k$, implemented by `IndexOfNonZeroColumn`.

Both these above oracles are unitary: they do not have any entangled ancilla/junk registers.


Note on sparsity: This bloq expects the user to provide the sparsity, as it is in general
difficult to compute the precise sparsity of the Kikuchi matrix efficiently. As long as the
provided number is at least the true sparsity, the algorithm will work as expected.
In case the provided sparsity is smaller, it is equivalent to making the remaining entries zero in the final block encoding.

#### Parameters
 - `inst`: the kXOR instance $\mathcal{I}$.
 - `ell`: Kikuchi parameter $\ell$.
 - `s`: sparsity, i.e. max number of non-zero entries in a row/column. 

#### Registers
 - `i`: integer in [2^N]
 - `k`: integer in [2^N] 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Theorem 4.17, proof para 4 (top of page 39).


In [ ]:
# The source declarations are imported in the setup cell.


### Example Instances

In [ ]:
inst = example_kxor_instance()
ell = 8
s = inst.brute_force_sparsity(ell)
kikuchi_nonzero_index = KikuchiNonZeroIndex(inst, ell, s=s)


#### Graphical Signature

In [ ]:
show_signature("kikuchi_nonzero_index", kikuchi_nonzero_index.signature)


### Call Graph

In [ ]:
kikuchi_nonzero_index_graph = kikuchi_nonzero_index.source_graph(
    root="KikuchiNonZeroIndex",
    forward="ColumnOfKthNonZeroEntry",
    reverse="IndexOfNonZeroColumn",
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                kikuchi_nonzero_index_graph.root,
                kikuchi_nonzero_index_graph.direct_calls,
            )
        )
    )
)
kikuchi_nonzero_index_sigma = kikuchi_nonzero_index_graph.leaf_counts
display(kikuchi_nonzero_index_sigma)


In [ ]:
# Declared QEC categories remain distinct from emitted physical primitives.
model_categories = kikuchi_nonzero_index.resource_gates()
assert model_categories.toffoli == 56
assert model_categories.cswap == 6144
assert model_categories.and_bloq == model_categories.measurement == 12828
display(model_categories)
